# German Podcast to English Podcast (Colab, Hugging Face only)

This notebook runs end-to-end on a Colab T4 GPU:
1. Read a German MP3 podcast
2. Transcribe with diarisation
3. Translate to English
4. Generate a multi-speaker English podcast
5. Download the translated MP3


## Model selection (for T4 and <60 minutes target)

- **ASR**: `openai/whisper-large-v3-turbo` (best quality/speed balance on T4 for long-form speech)
- **Diarisation**: `pyannote/speaker-diarization-3.1` (state-of-the-art open diarisation on Hugging Face)
- **Translation**: `facebook/nllb-200-distilled-1.3B` (highest-quality Hugging Face option that is still practical on T4)
- **TTS**: `coqui/XTTS-v2` (multi-speaker, voice cloning from reference audio, strong podcast suitability)

Before running: accept model terms on Hugging Face for `pyannote/speaker-diarization-3.1` and create a read token. Store it in Colab Secrets as `HF_TOKEN`.


In [ ]:
%pip -q install whisperx==3.1.5 pyannote.audio==3.1.1 transformers==4.46.3 accelerate==1.0.1 sentencepiece==0.2.0 TTS==0.22.0 pydub==0.25.1


In [ ]:
import torch
import whisperx
import pandas as pd
from pathlib import Path
from pydub import AudioSegment
from transformers import pipeline
from TTS.api import TTS
from google.colab import files, userdata

HF_TOKEN = userdata.get("HF_TOKEN")
uploaded = files.upload()
audio_path = next(iter(uploaded))
device = "cuda"


In [ ]:
audio = whisperx.load_audio(audio_path)
asr_model = whisperx.load_model(
    "openai/whisper-large-v3-turbo",
    device,
    compute_type="float16",
    language="de"
)
asr_result = asr_model.transcribe(audio, batch_size=16)
align_model, align_metadata = whisperx.load_align_model(language_code="de", device=device)
aligned_result = whisperx.align(asr_result["segments"], align_model, align_metadata, audio, device)
diarize_model = whisperx.DiarizationPipeline(
    model_name="pyannote/speaker-diarization-3.1",
    use_auth_token=HF_TOKEN,
    device=device
)
diarize_segments = diarize_model(audio)
diarized_result = whisperx.assign_word_speakers(diarize_segments, aligned_result)
segments_df = pd.DataFrame(diarized_result["segments"])[["start", "end", "speaker", "text"]]
segments_df["speaker"] = segments_df["speaker"].fillna("SPEAKER_00")


In [ ]:
translator = pipeline(
    task="translation",
    model="facebook/nllb-200-distilled-1.3B",
    src_lang="deu_Latn",
    tgt_lang="eng_Latn",
    device=0,
    torch_dtype=torch.float16
)
translated = translator(segments_df["text"].tolist(), batch_size=8, max_length=512)
segments_df["text_en"] = [item["translation_text"] for item in translated]
segments_df.to_csv("podcast_transcript_en.csv", index=False)


In [ ]:
work_dir = Path("translated_segments")
work_dir.mkdir(exist_ok=True)
source_audio = AudioSegment.from_file(audio_path)
speaker_refs = segments_df.sort_values("start").drop_duplicates("speaker")[["speaker", "start", "end"]]
speaker_refs["ref_path"] = [work_dir / f"{speaker}_ref.wav" for speaker in speaker_refs["speaker"]]
for row in speaker_refs.itertuples(index=False):
    source_audio[int(row.start * 1000):int(row.end * 1000)].export(row.ref_path, format="wav")
speaker_ref_map = dict(zip(speaker_refs["speaker"], speaker_refs["ref_path"]))
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)
segment_paths = []
for row in segments_df.itertuples(index=False):
    segment_path = work_dir / f"seg_{int(row.start * 1000):09d}.wav"
    tts.tts_to_file(
        text=row.text_en,
        speaker_wav=str(speaker_ref_map[row.speaker]),
        language="en",
        file_path=str(segment_path)
    )
    segment_paths.append(segment_path)


In [ ]:
translated_audio = AudioSegment.silent(duration=0)
cursor_ms = 0
for row, segment_path in zip(segments_df.itertuples(index=False), segment_paths):
    start_ms = int(row.start * 1000)
    translated_audio = translated_audio + AudioSegment.silent(duration=max(start_ms - cursor_ms, 0))
    segment_audio = AudioSegment.from_wav(segment_path)
    translated_audio = translated_audio + segment_audio
    cursor_ms = len(translated_audio)
translated_audio.export("translated_podcast_en.mp3", format="mp3")
files.download("translated_podcast_en.mp3")
